# PACE MiniGrid on one Kaggle GPU

Set **Accelerator** to GPU and enable **Internet**. This notebook tests the 145-interaction cap on four small PACE batches. A fresh process resumes from batch two, and the notebook checks that the run stops at 144 interactions with one step left. Checkpoints and evaluation JSON go under `/kaggle/working/pace`.

In [ ]:
import os
import shutil
import subprocess
import sys
import tempfile
import zipfile
from pathlib import Path

WORK = Path('/kaggle/working')
REPO_URL = 'https://github.com/SimforianusJonathan/PACE-Research.git'
REPO_REF = '178a8e85813ea94f96077aaece05f3ebf0987b8f'
REPO = WORK / f"PACE-Research-{REPO_REF.replace('/', '-')[:12]}"
SOURCE = ''  # Optional path to an uploaded source directory or Git archive ZIP.
RUNTIME = Path('/tmp/pace-kaggle')

if not REPO.exists():
    if SOURCE:
        source = Path(SOURCE)
        if source.is_dir():
            shutil.copytree(source, REPO)
        else:
            with zipfile.ZipFile(source) as archive:
                archive.extractall(REPO)
    else:
        subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
        subprocess.run(['git', 'checkout', '--detach', REPO_REF], cwd=REPO, check=True)

if not (REPO / 'scripts/setup_kaggle.sh').is_file():
    raise RuntimeError('This source lacks Kaggle support. Use a Git ref or source archive containing this notebook.')
if (REPO / '.git').exists():
    subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO, check=True)

# Use Kaggle's unmodified driver environment for this diagnostic.
smi = shutil.which('nvidia-smi')
if not smi or subprocess.run([smi, '-L'], check=False).returncode != 0:
    print('nvidia-smi did not list a GPU. Checking the CUDA driver directly.')

visible_device = os.environ.get('CUDA_VISIBLE_DEVICES', '0').split(',', 1)[0]
env = {**os.environ, 'PACE_RUNTIME_DIR': str(RUNTIME), 'CUDA_VISIBLE_DEVICES': visible_device,
       'JAX_PLATFORMS': 'cuda', 'XLA_PYTHON_CLIENT_PREALLOCATE': 'false',
       'MPLBACKEND': 'Agg', 'PYTHONUNBUFFERED': '1'}
# Keep mounted driver libraries while letting the wheels provide the CUDA toolkit.
paths = ['/usr/local/nvidia/lib64', *os.environ.get('LD_LIBRARY_PATH', '').split(os.pathsep)]
driver_paths = dict.fromkeys(path for path in paths if Path(path).is_absolute()
                            and Path(path).name not in {'compat', 'stubs'}
                            and (Path(path) / 'libcuda.so.1').is_file())
if driver_paths:
    env['LD_LIBRARY_PATH'] = os.pathsep.join(driver_paths)
else:
    env.pop('LD_LIBRARY_PATH', None)

driver_probe = '''
import ctypes
cuda = ctypes.CDLL('libcuda.so.1')
status = cuda.cuInit(0)
if status != 0:
    raise RuntimeError(f'CUDA driver initialization failed (cuInit={status})')
count = ctypes.c_int()
status = cuda.cuDeviceGetCount(ctypes.byref(count))
if status != 0 or count.value < 1:
    raise RuntimeError(f'No visible CUDA device (status={status}, count={count.value})')
'''
try:
    subprocess.run([sys.executable, '-c', driver_probe], env=env, check=True)
except subprocess.CalledProcessError:
    raise RuntimeError('Kaggle has no usable CUDA device for this run. Select a GPU accelerator, '
                       'then restart the session or save a new GPU-enabled version.') from None

In [ ]:
def run_logged(command, log_path):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print(f'Log: {log_path}', flush=True)
    with log_path.open('w', encoding='utf-8') as log, subprocess.Popen(
        command, cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding='utf-8', errors='replace',
    ) as process:
        for line in process.stdout:
            log.write(line)
            log.flush()
            print(line, end='', flush=True)
        code = process.wait()
    if code:
        raise RuntimeError(f'{log_path.stem} failed with exit code {code}. Full log: {log_path}')

run_logged(['bash', 'scripts/setup_kaggle.sh'], WORK / 'pace' / 'setup.log')
PACE_PYTHON = str(RUNTIME / 'venv/bin/python')

probe = '''
import importlib.metadata as metadata
import json
import jax
import jax.numpy as jnp
from jaxued.environments import Maze
from train.pace_minigrid import MiniGridConfig
if getattr(MiniGridConfig.smoke(), 'protocol', None) != 'algorithm1':
    raise RuntimeError('Select a repository revision with the algorithm1 training protocol.')
devices = jax.local_devices(backend='gpu')
assert len(devices) == 1, devices
result = jax.jit(lambda x: x @ x)(jnp.ones((16, 16))).block_until_ready()
assert result.device.platform == 'gpu' and float(result[0, 0]) == 16.0
pin = '0f8f1284677375b889e4f13a32c9617cd009f8c4'
origin = json.loads(metadata.distribution('jaxued').read_text('direct_url.json'))
assert origin['vcs_info']['commit_id'] == pin
print({'device': str(devices[0]), 'jax': jax.__version__, 'jaxued_commit': pin, 'environment': Maze.__name__})
'''
run_logged([PACE_PYTHON, '-c', probe], WORK / 'pace' / 'gpu-probe.log')

In [ ]:
import json

runs = WORK / 'pace'
runs.mkdir(exist_ok=True)
run = Path(tempfile.mkdtemp(prefix='smoke-', dir=runs))
base = [PACE_PYTHON, '-m', 'train.pace_minigrid']
run_logged(base + ['smoke', '--device', 'gpu', '--output-dir', str(run),
                   '--protocol', 'algorithm1', '--budget', 'env_steps', '--max-env-steps', '145',
                   '--checkpoint-interval', '2', '--stop-after', '2'], run / 'smoke.log')
run_logged(base + ['resume', '--device', 'gpu', '--output-dir', str(run),
                   '--checkpoint', str(run / 'checkpoints/step_00000002.msgpack'),
                   '--checkpoint-interval', '2'], run / 'resume.log')
final = run / 'checkpoints/step_00000004.msgpack'
assert final.is_file(), final
lines = (run / 'resume.log').read_text(encoding='utf-8').splitlines()
summary = next((json.loads(line) for line in reversed(lines) if line.startswith('{')), {})
expected = {
    'rollout_batches': 4, 'ppo_batches': 4, 'optimizer_steps': 4, 'main_collection_batches': 8,
    'env_steps': 144, 'prefill_env_steps': 16, 'ppo_env_steps': 64, 'scoring_env_steps': 64,
    'shared_env_steps': 0, 'budget_remaining': 1, 'stop_reason': 'env_steps_budget',
}
if any(summary.get(key) != value for key, value in expected.items()):
    raise RuntimeError(f'Unexpected smoke counters: {summary}')
print('Smoke passed and checkpoint resumed:', final)

In [ ]:
run_logged(base + ['eval', '--device', 'gpu', '--checkpoint', str(final),
                   '--suite', 'reference', '--levels', 'Maze3', '--episodes', '2',
                   '--horizon', '16', '--output-dir', str(run / 'eval')], run / 'eval.log')